# matvec — worked example 1: Project a Vector onto a Subspace via Matvec

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `matvec`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Multiplying a matrix `A: (M, N)` by a vector `x: (N,)` using `A @ x` produces a vector of shape `(M,)`. This is the fundamental linear map: `A` transforms `x` from an `N`-dimensional space to an `M`-dimensional space. When `A` contains orthonormal rows, this computes the projection of `x` onto each of those row directions.

## Worked solution

**Step 1 — create a weight matrix and input vector.**
We build `W: (3, 4)` and `v: (4,)`. The matrix has more columns than rows — it maps from 4-D space to 3-D space.

**Step 2 — apply the matvec.**
We compute `out = W @ v` using the `@` operator. PyTorch detects that `v` is 1-D and produces a 1-D output of shape `(3,)`, not `(3, 1)`. This is the key behavior: passing a 1-D vector directly keeps the output 1-D.

**Step 3 — verify the output shape.**
We confirm `out.shape == (3,)`. If we had inadvertently called `W @ v.unsqueeze(-1)`, the output would be `(3, 1)`. We demonstrate this contrast.

**Step 4 — check one entry manually.**
We verify that `out[0] == W[0] @ v` (dot product of the first row of W with v), confirming the matvec computes each output coordinate as a row-dot-vector.

In [ ]:
import torch as t

t.manual_seed(3)
W = t.randn(3, 4)   # weight matrix: 4-D input -> 3-D output
v = t.randn(4)      # 1-D input vector

# Matvec: (3,4) @ (4,) -> (3,)  — stays 1-D
out = W @ v
print(f"W shape:   {W.shape}")
print(f"v shape:   {v.shape}")
print(f"out shape: {out.shape}  <- must be (3,), NOT (3,1)")
print(f"out value: {out.round(decimals=4)}")

# Contrast: using unsqueeze gives (3,1)
out_col = W @ v.unsqueeze(-1)
print(f"\nWith unsqueeze: out_col shape: {out_col.shape}  <- (3,1)")
print(f"Values match after squeeze: {t.allclose(out, out_col.squeeze(-1))}")

# Verify one entry manually: out[1] = W[1] . v
dot_check = (W[1] * v).sum()
print(f"\nout[1] = {out[1]:.4f}")
print(f"W[1].v = {dot_check:.4f}")
print(f"Match:  {t.isclose(out[1], dot_check)}")